# 04 — Query 4: Spatial Analysis & Coastal Hotspot Clustering

- **Owner:** Louis Le Forestier (`@louislefo`)
- **Course Rubric:** Query 4 `[2 pts]` (reusing Lab 8 `GeoPandas`, `shapely`, `sjoin`, `KMeans`, and `OPTICS`)

In [1]:
# Import libraries and project functions
import os
import sys
import warnings
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import load_shark_data, clean_shark_data
from src.indicators import geocode_coastal_areas, query_spatial_clustering

warnings.filterwarnings('ignore')
clean_sharks = clean_shark_data(load_shark_data('../data/attacks.csv'))
GEO_CACHE = '../data/locations_geocoded.csv'
SHP_PATH = '../data/countries_shp/ne_110m_admin_0_countries.shp'

## Query 4 — Coastal Clustering with GeoPandas, K-means and OPTICS

Assign approximate coordinates from reported coastal areas. Keep country-only coordinates in the cache for coverage reporting and exclude them from clustering. When the Natural Earth shapefile is available, join area points to country polygons and retain a single row per incident.

Compare six K-means clusters with the denser groups found by OPTICS. The table and map below describe clusters of reported incidents using approximate coastal locations.

In [2]:
# Build/load coastal coordinates cache and run GeoPandas + K-Means + OPTICS clustering
geo_coords = geocode_coastal_areas(clean_sharks, cache_path=GEO_CACHE, force_refresh=True)
q4 = query_spatial_clustering(clean_sharks, geo_coords, shp_path=SHP_PATH, n_clusters=6)

print(f"Area-level geocoding coverage: {q4['coverage_pct']}% ({len(q4['geo_df'])} / {len(clean_sharks)} incidents)")
print(f"Country-only points excluded: {q4['country_centroid_count']}")
print(f"GeoPandas sjoin land-polygon matches: {q4['sjoin_matched']} points")
print('\n--- K-Means Spatial Cluster Profiles (Dynamic Centroid Labels) ---')
display(q4['cluster_summary'])
q4['fig_map'].show()

Area-level geocoding coverage: 76.0% (3065 / 4033 incidents)
Country-only points excluded: 790
GeoPandas sjoin land-polygon matches: 2434 points

--- K-Means Spatial Cluster Profiles (Dynamic Centroid Labels) ---


,Cluster_Name,Total_Incidents,Fatal_Count,Mean_Lat,Mean_Lon,Dominant_Activity,Fatality_Rate_Pct
0,"US Atlantic, Florida & Caribbean (United State...",1294,41,28.96,-81.15,Surfing,3.17
1,Australia & South-West Pacific (Australia / Ne...,760,82,-32.92,144.85,Surfing,10.79
2,Southern Africa & Indian Ocean (South Africa /...,434,54,-32.39,25.50,Surfing,12.44
3,Pacific & US West Coast / Hawaii (United State...,269,14,35.25,-119.92,Surfing,5.20
4,Pacific & US West Coast / Hawaii (United States),222,21,20.80,-156.33,Surfing,9.46
5,South America & Tropical Americas (Brazil),86,28,-9.96,-35.97,Surfing,32.56


In [3]:
# Inspect OPTICS density-based spatial clustering comparison (Lab 8)
print('--- OPTICS Density Cluster Summary ---')
display(q4['optics_summary'])
q4['fig_optics_map'].show()

--- OPTICS Density Cluster Summary ---


,OPTICS_Label,Total_Incidents,Top_Country
0,Dense Hotspot #2,919,United States
1,Noise / Isolated Coast (-1),458,United States
2,Dense Hotspot #9,243,Australia
3,Dense Hotspot #0,233,United States
4,Dense Hotspot #3,222,United States
5,Dense Hotspot #10,159,Australia
6,Dense Hotspot #4,158,South Africa
7,Dense Hotspot #7,142,Australia
8,Dense Hotspot #8,140,Australia
9,Dense Hotspot #6,135,South Africa
